# Step 4 of RAG: Passing Context to the Gemini / Groq LLM for Answering**RAG** (Retrieval-Augmented Generation) pipeline recap:1. Document Loading & Chunking (Step 1)2. Storing chunks in a Vector Database (Step 2)3. Retrieving relevant context for a user query (Step 3)4. **Passing the retrieved context to an LLM to generate the final answer** ← *this notebook*We use two **free-tier** LLM providers (you can use either one, or both):| Provider | Free API key from | Model family ||---|---|---|| **Groq** | https://console.groq.com/keys | Llama / GPT-OSS (very fast) || **Google Gemini** | https://aistudio.google.com/apikey | Gemini Flash |> **You only need ONE key.** If you skip a provider, the notebook simply skips that part — no errors.> **Security tip:** never paste API keys directly into notebook code or share a notebook with a key inside it.### What you will learn1. Safely entering API keys in Colab2. Rebuilding the retrieval pipeline (Steps 1-3) in one condensed block3. Writing a **grounded system prompt** that stops hallucination4. Calling **Groq** with the retrieved context5. Calling **Gemini** with the retrieved context6. Building one `rag_answer()` function that works with either provider7. Testing: in-scope question, multi-chunk question, and out-of-scope question8. Comparing both LLMs on the same question

### Setup — Install Required Libraries

In [ ]:
!pip install -q langchain-text-splitters chromadb sentence-transformers groq google-genai

## 1. Enter Your API Keys SafelyThis cell looks for a key in three places, in order:1. An environment variable2. Colab **Secrets** (the key icon in the left sidebar) — the safest option3. A hidden input box (`getpass`) — just press **Enter** to skip a providerThe names to use are `GROQ_API_KEY` and `GEMINI_API_KEY`.

In [ ]:
import osimport getpassdef get_key(name):    key = os.environ.get(name, "")    if not key:        try:            from google.colab import userdata            key = userdata.get(name) or ""        except Exception:            key = ""    if not key:        key = getpass.getpass(f"Enter {name} (press Enter to skip): ").strip()    return keyGROQ_API_KEY = get_key("GROQ_API_KEY")GEMINI_API_KEY = get_key("GEMINI_API_KEY")print("Groq key provided:  ", bool(GROQ_API_KEY))print("Gemini key provided:", bool(GEMINI_API_KEY))if not GROQ_API_KEY and not GEMINI_API_KEY:    print("\nNo keys provided. The retrieval parts will still run, but LLM answers will be skipped.")

## 2. Recap — Chunk, Store and Retrieve (Steps 1-3 Condensed)Same pipeline as the earlier notebooks, condensed so this notebook runs on its own.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitterimport chromadbfrom chromadb.utils import embedding_functionsdocuments = {    "ai_overview.txt": """Artificial Intelligence: An OverviewArtificial Intelligence, or AI, is the field of computer science focused on building systemsthat can perform tasks which normally require human intelligence. These tasks includeunderstanding language, recognizing images, making decisions, and learning from experience.Machine Learning is a subset of AI where systems learn patterns from data instead of followinghardcoded rules. Instead of programming every decision by hand, we feed the system many examples,and it learns the underlying pattern on its own.Deep Learning takes Machine Learning further by using neural networks with many layers. Theselayered networks can automatically discover complex patterns in large amounts of data, such asrecognizing faces in photos or understanding the meaning of a sentence.Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with alanguage model. Instead of relying only on what a language model memorized during training, RAGfirst retrieves relevant, up-to-date information from a document collection, then passes thatinformation to the model so it can generate a more accurate, grounded answer.""",    "embeddings_in_practice.txt": """Embeddings and Vector Search in PracticeModern search engines increasingly rely on embeddings rather than simple keyword matching.An embedding model converts a search query into the same vector space as the stored documents,allowing the system to find results that match the meaning of the query, not just its exact words.This is especially powerful for customer support systems, where a user might phrase a question verydifferently from how it appears in a help article, yet still expect to find the right answer.Vector databases such as ChromaDB store these embeddings and allow fast similarity search acrossthousands or millions of documents at once."""}splitter = RecursiveCharacterTextSplitter(    chunk_size=250, chunk_overlap=40, separators=["\n\n", "\n", ". ", " ", ""])client = chromadb.Client()hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")collection = client.create_collection(name="rag_chunks", embedding_function=hf_embedder)for source, text in documents.items():    doc_chunks = splitter.split_text(text)    collection.add(        documents=doc_chunks,        metadatas=[{"source": source, "chunk_index": i} for i in range(len(doc_chunks))],        ids=[f"{source}_chunk_{i}" for i in range(len(doc_chunks))]    )print("Total chunks stored:", collection.count())

In [ ]:
def retrieve_context(query, n_results=4, similarity_threshold=0.25):    results = collection.query(        query_texts=[query],        n_results=n_results,        include=["documents", "metadatas", "distances"]    )    context_chunks = []    for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):        similarity = 1 - distance        if similarity >= similarity_threshold:            context_chunks.append({                "text": doc,                "source": meta["source"],                "chunk_index": meta["chunk_index"],                "similarity": round(similarity, 4)            })    return context_chunks# Quick check that retrieval workstest_chunks = retrieve_context("What is RAG?")print("Chunks retrieved:", len(test_chunks))for c in test_chunks:    print(f"  [{c['similarity']}] {c['source']} #{c['chunk_index']}")

## 3. Writing a Grounded PromptThe prompt is what turns a general-purpose LLM into a **RAG** system. Two parts matter:- **System prompt** — the rules: *answer only from the context, say "I don't know" otherwise, cite sources.*- **User prompt** — the retrieved context + the user's question.Setting a **low temperature** (0.1) makes the model stick to the facts instead of being creative.

In [ ]:
SYSTEM_PROMPT = (    "You are a helpful assistant that answers questions using ONLY the provided context. "    "If the answer is not in the context, reply exactly: \"I don't know based on the provided documents.\" "    "Keep the answer short and clear. At the end, mention the source file name(s) you used.")def build_user_prompt(query, context_chunks):    context_text = "\n\n".join(        f"[Source: {c['source']}, chunk {c['chunk_index']}]\n{c['text']}"        for c in context_chunks    )    return f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"sample_chunks = retrieve_context("How does RAG improve a language model?")print(build_user_prompt("How does RAG improve a language model?", sample_chunks))

**Discussion Point:** The LLM never "searches" anything itself. It only sees the text we paste into theprompt — so the quality of the answer depends directly on the quality of the retrieved chunks.

## 4. Calling Groq (Llama / GPT-OSS models)Groq exposes an OpenAI-style chat API. We try a short list of models in order, so if one isrenamed or temporarily unavailable, the next one is used automatically instead of crashing.

In [ ]:
from groq import GroqGROQ_MODELS = ["llama-3.3-70b-versatile", "llama-3.1-8b-instant", "openai/gpt-oss-20b"]def ask_groq(user_prompt):    if not GROQ_API_KEY:        return "[Groq skipped: no API key provided]"    groq_client = Groq(api_key=GROQ_API_KEY)    last_error = None    for model_name in GROQ_MODELS:        try:            response = groq_client.chat.completions.create(                model=model_name,                messages=[                    {"role": "system", "content": SYSTEM_PROMPT},                    {"role": "user", "content": user_prompt}                ],                temperature=0.1,                max_tokens=600            )            answer = response.choices[0].message.content            if answer:                return answer.strip()        except Exception as e:            last_error = e    return f"[Groq error: {last_error}]"question = "How does RAG improve a language model?"chunks = retrieve_context(question)print("Groq answer:\n")print(ask_groq(build_user_prompt(question, chunks)))

## 5. Calling Google GeminiWe use the official `google-genai` SDK. Again, a short model list gives us an automatic fallback.`system_instruction` plays the same role as the system message we used with Groq.

In [ ]:
from google import genaifrom google.genai import typesGEMINI_MODELS = ["gemini-2.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash-lite"]def ask_gemini(user_prompt):    if not GEMINI_API_KEY:        return "[Gemini skipped: no API key provided]"    gemini_client = genai.Client(api_key=GEMINI_API_KEY)    last_error = None    for model_name in GEMINI_MODELS:        try:            response = gemini_client.models.generate_content(                model=model_name,                contents=user_prompt,                config=types.GenerateContentConfig(                    system_instruction=SYSTEM_PROMPT,                    temperature=0.1,                    max_output_tokens=1500                )            )            if response.text:                return response.text.strip()        except Exception as e:            last_error = e    return f"[Gemini error: {last_error}]"print("Gemini answer:\n")print(ask_gemini(build_user_prompt(question, chunks)))

## 6. One Function for the Full RAG Pipeline: `rag_answer()`This ties everything together: **retrieve → build prompt → ask the LLM → show sources**.If retrieval finds nothing relevant, we skip the LLM completely — that saves tokens and avoids amade-up answer.

In [ ]:
def rag_answer(query, provider="groq", show_sources=True):    context_chunks = retrieve_context(query)    if not context_chunks:        return "I don't know based on the provided documents. (No relevant chunks were retrieved.)"    user_prompt = build_user_prompt(query, context_chunks)    if provider == "gemini":        answer = ask_gemini(user_prompt)    else:        answer = ask_groq(user_prompt)    if show_sources:        sources = sorted({f"{c['source']} #{c['chunk_index']}" for c in context_chunks})        answer += "\n\nRetrieved from: " + ", ".join(sources)    return answer# Pick the provider you have a key fordefault_provider = "groq" if GROQ_API_KEY else "gemini"print("Using provider:", default_provider)

## 7. Testing the Full RAG Pipeline

**Test 1 — A simple in-scope question**

In [ ]:
q1 = "What is deep learning?"print("Question:", q1, "\n")print(rag_answer(q1, provider=default_provider))

**Test 2 — A question that needs information from more than one document**

In [ ]:
q2 = "How do embeddings and vector databases help RAG find useful information?"print("Question:", q2, "\n")print(rag_answer(q2, provider=default_provider))

**Test 3 — An out-of-scope question**Our documents say nothing about cooking. A good RAG system must admit that instead of guessing.

In [ ]:
q3 = "What is the best recipe for chocolate cake?"print("Question:", q3, "\n")print(rag_answer(q3, provider=default_provider))

**Discussion Point:** Test 3 is the most important one. Without the similarity threshold and thegrounded system prompt, a plain LLM would happily invent a cake recipe. RAG's job is to stayinside the knowledge base.

## 8. Comparing Groq and Gemini on the Same QuestionIf you provided both keys, this cell shows both answers side by side. If you gave only one key,the other provider is politely skipped.

In [ ]:
compare_q = "Why is RAG more reliable than asking a language model directly?"print("Question:", compare_q, "\n")compare_chunks = retrieve_context(compare_q)compare_prompt = build_user_prompt(compare_q, compare_chunks)print("=" * 25, "GROQ", "=" * 25)print(ask_groq(compare_prompt))print("\n" + "=" * 24, "GEMINI", "=" * 24)print(ask_gemini(compare_prompt))

## 9. Interactive Q&A (Optional)Ask your own questions about the documents. Leave the box empty and press Enter to stop.

In [ ]:
while True:    user_question = input("Ask a question (or press Enter to quit): ").strip()    if not user_question:        print("Goodbye!")        break    print("\n" + rag_answer(user_question, provider=default_provider) + "\n")

## Key Takeaways (Recap for Students)| Concept | What We Did ||---|---|| API keys | Read from Colab Secrets or a hidden `getpass` box — never hardcoded || System prompt | Told the LLM to answer *only* from context and admit when it doesn't know || User prompt | Combined the retrieved chunks + the question, labeled with source names || Low temperature | `temperature=0.1` keeps answers factual and consistent || Groq | `groq` SDK, `chat.completions.create(...)`, fast open models || Gemini | `google-genai` SDK, `models.generate_content(...)` with `system_instruction` || Model fallback | Tried a list of models so a renamed/unavailable model doesn't crash the notebook || Threshold guard | Skipped the LLM entirely when nothing relevant was retrieved |**The complete RAG pipeline:** Load → Chunk → Embed → Store (Steps 1-2) → Retrieve (Step 3) →Generate a grounded answer (Step 4).### Practice Exercises1. Add your own `.txt` content as a third document in the `documents` dictionary and ask questions about it.2. Change `temperature` from `0.1` to `1.0` and compare how the answers change.3. Change the system prompt so the answer is returned as exactly **3 bullet points**.4. Set `n_results=1` in `retrieve_context()` and test Test 2 again — what information is lost?

## Practice Exercises — Worked Answers

**Name:** N.Geethika  
**Reg No:** 99230040381

Run the original setup cells first. These examples use the notebook's `documents`, `retrieve_context()`, and LLM generation setup; adjust only if your existing cells use different names.

### Exercise 1 — Add a third `.txt` document to `documents`

Add your own text as another entry, then ask a question that can only be answered from the new document.

In [ ]:
# Add a third document to the existing documents dictionary.
documents['third_document.txt'] = '''A good study routine uses short focused sessions, regular breaks,
and a weekly review of difficult topics. Practice questions help identify gaps in understanding.'''

# Ask a question about the new content using the notebook's existing flow.
question_1 = 'How can practice questions help a student?'
context_1 = retrieve_context(question_1)
print('Retrieved context:', context_1)
# If your notebook uses a separate generate_answer(question, context) function:
# print(generate_answer(question_1, context_1))

### Exercise 2 — Change `temperature` from `0.1` to `1.0`

Generate an answer to the same question at both settings. Keep the model and retrieved context unchanged so the comparison is fair.

In [ ]:
question_2 = 'Summarize the key benefits described in the retrieved context.'
context_2 = retrieve_context(question_2)

# Adapt the call below to the generation function used in your original notebook.
# answer_low = generate_answer(question_2, context_2, temperature=0.1)
# answer_high = generate_answer(question_2, context_2, temperature=1.0)
# print('Temperature 0.1:\n', answer_low)
# print('Temperature 1.0:\n', answer_high)

**Expected comparison:** temperature 0.1 generally produces more predictable, focused wording. Temperature 1.0 allows more variation in phrasing and emphasis. The exact answers depend on the model and context.

### Exercise 3 — Require exactly 3 bullet points

Change the system prompt so the generated answer has exactly three bullets. Use the updated prompt in your existing LLM call.

In [ ]:
system_prompt = '''Answer the user's question using only the provided context.
If the context does not contain the answer, say that the information is not available.
Return exactly 3 bullet points. Each bullet must be concise and relevant.
Do not add an introduction or conclusion outside the 3 bullets.'''

# Pass system_prompt to the same LLM generation call used in the notebook.
# answer = generate_answer(question, context, system_prompt=system_prompt, temperature=0.1)
# print(answer)

### Exercise 4 — Set `n_results=1` and repeat Test 2

Update `retrieve_context()` so its ChromaDB query asks for only one result, then repeat Test 2 and compare the answer with the original multi-chunk context.

In [ ]:
# In retrieve_context(), change the ChromaDB query to:
# results = collection.query(query_texts=[query], n_results=1, include=['documents', 'distances'])

# Re-run the exact Test 2 question from the original notebook:
# context_one = retrieve_context(test_2_question)
# answer_one = generate_answer(test_2_question, context_one, temperature=0.1)
# print('One-result context:', context_one)
# print('Answer with n_results=1:', answer_one)

**What information may be lost:** facts that appear only in the second or later most relevant chunks will not be included in the context. The answer may become shorter or omit supporting details, exceptions, or related facts. The exact loss depends on the Test 2 query and stored documents.

**Note:** The original notebook's exact generation function signature and Test 2 question should be reused. Lines shown as comments are integration examples where the exact function arguments must match your setup cells.